# HerWILL Safe Social Media Datathon 2026 — Harmful Content Classification

**Task:** Multi-class text classification of Bangla / English / Banglish (code-mixed) social media
text into three classes:

| Label | Meaning |
|---|---|
| `0` | Explicitly Toxic / Hateful |
| `1` | Subtly Toxic |
| `2` | Non-toxic / Neutral |

**Metric:** Macro F1 Score

$$F_1^{macro} = \frac{1}{C}\sum_{c=1}^{C}\frac{2P_cR_c}{P_c+R_c}$$

**Submission format:**
```
id,y_pred
0,1
1,0
2,2
...
```

## Methodology overview

This notebook trains and compares **three independent approaches**, then combines the best
transformer models into a **soft-voting ensemble** for the final submission:

1. **Sentence-embedding + classical ML baseline** — `paraphrase-multilingual-MiniLM-L12-v2`
   (SentenceTransformers) embeddings feeding a Logistic Regression / Linear SVM / LightGBM head.
2. **BanglaBERT fine-tuning** — `csebuetnlp/banglabert`, fine-tuned end-to-end for 3-class
   classification. Strong for Bangla-heavy and code-mixed text.
3. **mBERT fine-tuning** — `bert-base-multilingual-cased`, fine-tuned end-to-end. Strong general
   multilingual coverage (English + Bangla + code-mixed Banglish).
4. **Ensemble** — soft-voting (averaged softmax probabilities) across the fine-tuned transformer
   models, weighted by validation Macro F1.

Every stage reports **Macro F1 on a held-out validation split**, plots training/validation loss,
and the final cell writes `submission.csv` in the exact required format.

> **Only the dataset provided by the competition is used.** No external labeled data is used
> anywhere in this notebook, per the competition rules. `banglabert` and
> `paraphrase-multilingual-MiniLM-L12-v2` are used only as *pre-trained* backbones (allowed by the
> rules), fine-tuned exclusively on the competition's own training data.

## How to run this on Kaggle
1. Attach the competition dataset to this notebook (Add Data → the HerWILL competition dataset).
2. Turn on a **GPU accelerator** (Settings → Accelerator → GPU T4 x2 or P100).
3. Run all cells top to bottom (`Run All`). The notebook auto-detects the train/test CSV files and
   their id / text / label columns — if auto-detection ever picks the wrong file/column, edit the
   `CONFIG` cell (Section 1) and re-run.
4. The final cell writes `/kaggle/working/submission.csv`, ready to submit.


## 0. Setup — install & import packages

In [ ]:
# Packages that are not pre-installed on Kaggle images.
# (transformers / torch / scikit-learn / pandas / numpy are already present on Kaggle.)
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install([
    "sentence-transformers>=2.7.0",
    "lightgbm",
    "emoji",
])
print("Install step complete.")


In [ ]:
import os
import re
import json
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Configuration & data loading

Kaggle mounts competition data under `/kaggle/input/<competition-name>/`. Since the exact
folder / file names depend on the competition slug, the cell below **auto-discovers** every CSV
under `/kaggle/input`, then heuristically figures out which one is train (has a label column) and
which is test (has an `id` column but no label), and which columns hold the id / text / label.

If auto-detection ever guesses wrong for your copy of the dataset, just hard-code
`TRAIN_PATH`, `TEST_PATH`, `TEXT_COL`, `LABEL_COL`, `ID_COL` in the `CONFIG` dict below and
re-run — everything downstream reads from `CONFIG`, not from the detection logic.

In [ ]:
# ---- CONFIG -----------------------------------------------------------
# Leave values as None to auto-detect. Override any field manually if needed.
CONFIG = {
    "TRAIN_PATH": None,
    "TEST_PATH": None,
    "ID_COL": None,
    "TEXT_COL": None,
    "LABEL_COL": None,
    "NUM_CLASSES": 3,
    "MAX_LEN": 128,
    "VAL_SIZE": 0.15,
}

ID_CANDIDATES = ["id", "ID", "Id", "index", "Index", "sample_id"]
TEXT_CANDIDATES = ["text", "Text", "sentence", "Sentence", "comment", "Comment",
                    "content", "Content", "post", "Post", "caption", "Caption",
                    "status", "Status", "headline", "Headline", "tweet", "Tweet"]
LABEL_CANDIDATES = ["label", "Label", "labels", "target", "Target", "class", "Class",
                     "category", "Category", "y", "y_pred", "toxicity", "toxic_level"]

def find_col(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

all_csvs = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print(f"Found {len(all_csvs)} CSV file(s) under /kaggle/input:")
for p in all_csvs:
    print(" -", p)

if not all_csvs:
    raise FileNotFoundError(
        "No CSV files found under /kaggle/input. Attach the competition dataset "
        "(Add Data) and re-run, or set CONFIG['TRAIN_PATH'] / CONFIG['TEST_PATH'] manually."
    )

train_path, test_path = CONFIG["TRAIN_PATH"], CONFIG["TEST_PATH"]

if train_path is None or test_path is None:
    scored = []
    for p in all_csvs:
        try:
            head = pd.read_csv(p, nrows=5)
        except Exception:
            continue
        has_label = find_col(head.columns, LABEL_CANDIDATES) is not None
        has_id = find_col(head.columns, ID_CANDIDATES) is not None
        has_text = find_col(head.columns, TEXT_CANDIDATES) is not None
        scored.append((p, has_label, has_id, has_text, len(head.columns)))

    # Train candidate: has a label column and a text column.
    train_candidates = [s for s in scored if s[1] and s[3]]
    # Test candidate: has a text column, no label column (prefer id present).
    test_candidates = [s for s in scored if s[3] and not s[1]]

    if train_path is None and train_candidates:
        train_path = sorted(train_candidates, key=lambda s: -s[4])[0][0]
    if test_path is None and test_candidates:
        test_path = sorted(test_candidates, key=lambda s: -s[4])[0][0]

    # Fallback: filename heuristics if the above failed.
    if train_path is None:
        for p in all_csvs:
            if "train" in os.path.basename(p).lower():
                train_path = p
                break
    if test_path is None:
        for p in all_csvs:
            if "test" in os.path.basename(p).lower():
                test_path = p
                break

assert train_path is not None, "Could not auto-detect the training CSV — set CONFIG['TRAIN_PATH']."
assert test_path is not None, "Could not auto-detect the test CSV — set CONFIG['TEST_PATH']."
print("\nUsing TRAIN_PATH:", train_path)
print("Using TEST_PATH :", test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

ID_COL = CONFIG["ID_COL"] or find_col(test_df.columns, ID_CANDIDATES)
TEXT_COL = CONFIG["TEXT_COL"] or find_col(train_df.columns, TEXT_CANDIDATES)
LABEL_COL = CONFIG["LABEL_COL"] or find_col(train_df.columns, LABEL_CANDIDATES)

if ID_COL is None:
    # last resort: synthesize an id column for the test set
    test_df = test_df.reset_index().rename(columns={"index": "id"})
    ID_COL = "id"

assert TEXT_COL is not None, f"Could not detect text column. Train columns: {list(train_df.columns)}"
assert LABEL_COL is not None, f"Could not detect label column. Train columns: {list(train_df.columns)}"

print(f"\nDetected -> ID_COL='{ID_COL}'  TEXT_COL='{TEXT_COL}'  LABEL_COL='{LABEL_COL}'")

train_df = train_df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
test_df = test_df.rename(columns={TEXT_COL: "text", ID_COL: "id"})[["id", "text"]]

train_df = train_df.dropna(subset=["text"]).reset_index(drop=True)
train_df["label"] = train_df["label"].astype(int)
test_df["text"] = test_df["text"].fillna("")

print("\nTrain shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head()


## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

label_names = {0: "0: Explicit", 1: "1: Subtle", 2: "2: Neutral"}
counts = train_df["label"].map(label_names).value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Class distribution")
axes[0].set_ylabel("count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v, str(v), ha="center", va="bottom")

lengths = train_df["text"].astype(str).apply(lambda s: len(s.split()))
sns.histplot(lengths, bins=50, ax=axes[1], color="teal")
axes[1].set_title("Word-count distribution (train text)")
axes[1].set_xlabel("word count")

plt.tight_layout()
plt.show()

print(train_df["label"].value_counts(normalize=True).round(3))
print("\nMedian word count:", int(lengths.median()), " 95th pct:", int(lengths.quantile(0.95)))


In [ ]:
# Quick peek at a few examples per class
for lab in sorted(train_df["label"].unique()):
    print(f"--- label {lab} ---")
    sample = train_df[train_df["label"] == lab]["text"].sample(min(2, (train_df["label"]==lab).sum()), random_state=SEED)
    for t in sample:
        print(" •", str(t)[:150])
    print()


## 3. Text preprocessing

The dataset is Bangla + English + **Banglish** (romanized Bangla, code-mixed with English), and
noisy real-world social text. The cleaning function below is deliberately **conservative** — it
strips things that are pure noise (URLs, HTML entities, user handles, repeated punctuation,
excess whitespace) while **keeping** Bangla Unicode, emojis and punctuation intact, since emojis
and emphasis (e.g. "!!!", ALL CAPS) often carry toxicity signal and stripping them can hurt a
"subtle toxicity" classifier.

In [ ]:
import emoji

URL_RE = re.compile(r"http\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HTML_RE = re.compile(r"&[a-z]+;|<[^>]+>")
REPEAT_PUNCT_RE = re.compile(r"([!?.,])\1{2,}")
REPEAT_CHAR_RE = re.compile(r"(.)\1{3,}")  # e.g. "sooooo" -> "sooo"
MULTI_SPACE_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.strip()
    t = URL_RE.sub(" ", t)
    t = HTML_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = emoji.demojize(t, delimiters=(" :", ": "))  # keep emoji meaning as text token
    t = REPEAT_PUNCT_RE.sub(r"\1\1", t)
    t = REPEAT_CHAR_RE.sub(r"\1\1\1", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

train_df[["text", "clean_text"]].head(5)


In [ ]:
# Stratified train / validation split — used identically by every model below,
# so Macro F1 scores are directly comparable across approaches.
train_split, val_split = train_test_split(
    train_df,
    test_size=CONFIG["VAL_SIZE"],
    random_state=SEED,
    stratify=train_df["label"],
)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)
print("Train split:", train_split.shape, " Val split:", val_split.shape)
print(val_split["label"].value_counts(normalize=True).round(3))


## 4. Approach 1 — Multilingual sentence embeddings + classical ML

`paraphrase-multilingual-MiniLM-L12-v2` produces a single dense vector per sentence that already
captures cross-lingual semantic meaning (it was trained on 50+ languages, including Bangla and
English, aligned to the same embedding space). We freeze this encoder and train fast, light
classical classifiers on top — a strong, cheap baseline.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

def embed(texts, batch_size=64):
    return embedder.encode(
        list(texts),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

X_train_emb = embed(train_split["clean_text"])
X_val_emb = embed(val_split["clean_text"])
X_test_emb = embed(test_df["clean_text"])

y_train = train_split["label"].values
y_val = val_split["label"].values

print(X_train_emb.shape, X_val_emb.shape, X_test_emb.shape)


In [ ]:
results = {}

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=2000, C=2.0, class_weight="balanced", random_state=SEED)
logreg.fit(X_train_emb, y_train)
pred = logreg.predict(X_val_emb)
f1_lr = f1_score(y_val, pred, average="macro")
results["LogReg + MiniLM-embeddings"] = f1_lr
print(f"Logistic Regression  Macro F1: {f1_lr:.4f}")
print(classification_report(y_val, pred, digits=3))

# --- Linear SVM (calibrated for probabilities, needed for the ensemble later) ---
svm = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight="balanced", random_state=SEED), cv=3)
svm.fit(X_train_emb, y_train)
pred_svm = svm.predict(X_val_emb)
f1_svm = f1_score(y_val, pred_svm, average="macro")
results["LinearSVM + MiniLM-embeddings"] = f1_svm
print(f"\nLinear SVM  Macro F1: {f1_svm:.4f}")
print(classification_report(y_val, pred_svm, digits=3))


In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    objective="multiclass",
    num_class=CONFIG["NUM_CLASSES"],
    class_weight="balanced",
    random_state=SEED,
)
lgbm.fit(X_train_emb, y_train, eval_set=[(X_val_emb, y_val)], eval_metric="multi_logloss",
         callbacks=[lgb.early_stopping(30, verbose=False)])
pred_lgbm = lgbm.predict(X_val_emb)
f1_lgbm = f1_score(y_val, pred_lgbm, average="macro")
results["LightGBM + MiniLM-embeddings"] = f1_lgbm
print(f"LightGBM  Macro F1: {f1_lgbm:.4f}")
print(classification_report(y_val, pred_lgbm, digits=3))

best_embed_name = max(
    {"LogReg": f1_lr, "SVM": f1_svm, "LightGBM": f1_lgbm}.items(), key=lambda kv: kv[1]
)[0]
print("\nBest embedding-based classifier:", best_embed_name)


## 5. Approach 2 — Fine-tuning BanglaBERT

[`csebuetnlp/banglabert`](https://huggingface.co/csebuetnlp/banglabert) is an ELECTRA-style model
pre-trained specifically on Bangla text, which typically gives it an edge on Bangla-dominant and
Banglish content over generic multilingual models.

**Architecture:** `AutoModelForSequenceClassification` — the pretrained encoder + a linear
classification head (3 outputs), fine-tuned end-to-end.

**Key hyperparameters** (also used for mBERT in Section 6, for a fair comparison):
- max sequence length: 128
- learning rate: 2e-5, linear warmup (6% of steps) + linear decay
- batch size: 16 (train), 32 (eval)
- epochs: 4, with early stopping on validation Macro F1 (patience = 2)
- weight decay: 0.01
- optimizer: AdamW

In [ ]:
def build_hf_datasets(tokenizer, max_len):
    def tok_fn(batch):
        return tokenizer(batch["clean_text"], truncation=True, max_length=max_len)

    ds_train = HFDataset.from_pandas(train_split[["clean_text", "label"]])
    ds_val = HFDataset.from_pandas(val_split[["clean_text", "label"]])
    ds_test = HFDataset.from_pandas(test_df[["clean_text"]])

    ds_train = ds_train.map(tok_fn, batched=True)
    ds_val = ds_val.map(tok_fn, batched=True)
    ds_test = ds_test.map(tok_fn, batched=True)

    ds_train = ds_train.rename_column("label", "labels")
    ds_val = ds_val.rename_column("label", "labels")
    return ds_train, ds_val, ds_test


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": (preds == labels).mean(),
    }


def train_transformer(model_name, output_dir, max_len=CONFIG["MAX_LEN"], epochs=4,
                       lr=2e-5, train_bs=16, eval_bs=32):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=CONFIG["NUM_CLASSES"]
    )

    ds_train, ds_val, ds_test = build_hf_datasets(tokenizer, max_len)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_ratio=0.06,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()
    return trainer, tokenizer, ds_test


def extract_loss_curve(trainer):
    hist = trainer.state.log_history
    train_pts = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
    val_pts = [(h["epoch"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    return train_pts, val_pts


def plot_loss(train_pts, val_pts, title):
    plt.figure(figsize=(6, 4))
    if train_pts:
        xs, ys = zip(*train_pts)
        plt.plot(xs, ys, label="train loss", marker=".")
    if val_pts:
        xs, ys = zip(*val_pts)
        plt.plot(xs, ys, label="val loss", marker="o")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(title); plt.legend()
    plt.tight_layout(); plt.show()


def predict_probs(trainer, ds):
    out = trainer.predict(ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return probs


In [ ]:
BANGLABERT_NAME = "csebuetnlp/banglabert"

bangla_trainer, bangla_tok, bangla_ds_test = train_transformer(
    BANGLABERT_NAME, output_dir="/kaggle/working/banglabert_ckpt"
)

bangla_val_probs = predict_probs(bangla_trainer, bangla_trainer.eval_dataset)
bangla_val_pred = bangla_val_probs.argmax(axis=1)
f1_bangla = f1_score(y_val, bangla_val_pred, average="macro")
results["BanglaBERT (fine-tuned)"] = f1_bangla
print(f"\nBanglaBERT  Macro F1 (val): {f1_bangla:.4f}")
print(classification_report(y_val, bangla_val_pred, digits=3))

train_pts, val_pts = extract_loss_curve(bangla_trainer)
plot_loss(train_pts, val_pts, "BanglaBERT training/validation loss")


In [ ]:
cm = confusion_matrix(y_val, bangla_val_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["0:Explicit", "1:Subtle", "2:Neutral"],
            yticklabels=["0:Explicit", "1:Subtle", "2:Neutral"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("BanglaBERT — confusion matrix")
plt.tight_layout(); plt.show()


## 6. Approach 3 — Fine-tuning mBERT

`bert-base-multilingual-cased` is trained on 104 languages jointly. It tends to generalize better
on the **English-heavy and heavily code-mixed** posts than a Bangla-specialist model, which is
exactly why it's paired with BanglaBERT here — the two make complementary errors, which is ideal
for the ensemble in Section 7.

In [ ]:
MBERT_NAME = "bert-base-multilingual-cased"

mbert_trainer, mbert_tok, mbert_ds_test = train_transformer(
    MBERT_NAME, output_dir="/kaggle/working/mbert_ckpt"
)

mbert_val_probs = predict_probs(mbert_trainer, mbert_trainer.eval_dataset)
mbert_val_pred = mbert_val_probs.argmax(axis=1)
f1_mbert = f1_score(y_val, mbert_val_pred, average="macro")
results["mBERT (fine-tuned)"] = f1_mbert
print(f"\nmBERT  Macro F1 (val): {f1_mbert:.4f}")
print(classification_report(y_val, mbert_val_pred, digits=3))

train_pts, val_pts = extract_loss_curve(mbert_trainer)
plot_loss(train_pts, val_pts, "mBERT training/validation loss")


In [ ]:
cm = confusion_matrix(y_val, mbert_val_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["0:Explicit", "1:Subtle", "2:Neutral"],
            yticklabels=["0:Explicit", "1:Subtle", "2:Neutral"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("mBERT — confusion matrix")
plt.tight_layout(); plt.show()


## 7. Model comparison & ensemble

We combine BanglaBERT and mBERT by **soft-voting**: average their predicted class probabilities,
weighted by each model's validation Macro F1 (a better model gets proportionally more say). We
also try adding the best embedding-based classifier's probabilities into the vote, and keep
whichever combination scores highest on the validation split.

In [ ]:
comparison_df = pd.DataFrame(
    sorted(results.items(), key=lambda kv: -kv[1]), columns=["Model", "Val Macro F1"]
)
comparison_df["Val Macro F1"] = comparison_df["Val Macro F1"].round(4)
print(comparison_df.to_string(index=False))

plt.figure(figsize=(7, 4))
sns.barplot(data=comparison_df, x="Val Macro F1", y="Model", palette="crest")
plt.title("Validation Macro F1 by model")
plt.tight_layout(); plt.show()


In [ ]:
# Best embedding-based classifier's probabilities on the val split, for the optional 3-way ensemble.
embed_models = {"LogReg": logreg, "SVM": svm, "LightGBM": lgbm}
best_embed_clf = embed_models[best_embed_name]
embed_val_probs = best_embed_clf.predict_proba(X_val_emb)

def weighted_soft_vote(prob_list, weights):
    weights = np.array(weights) / np.sum(weights)
    stacked = np.stack(prob_list, axis=0)  # (n_models, n_samples, n_classes)
    return np.tensordot(weights, stacked, axes=(0, 0))

# 2-way ensemble: BanglaBERT + mBERT
ens2_probs = weighted_soft_vote([bangla_val_probs, mbert_val_probs], [f1_bangla, f1_mbert])
ens2_pred = ens2_probs.argmax(axis=1)
f1_ens2 = f1_score(y_val, ens2_pred, average="macro")

# 3-way ensemble: BanglaBERT + mBERT + best embedding classifier
ens3_probs = weighted_soft_vote(
    [bangla_val_probs, mbert_val_probs, embed_val_probs],
    [f1_bangla, f1_mbert, max(f1_lr, f1_svm, f1_lgbm)],
)
ens3_pred = ens3_probs.argmax(axis=1)
f1_ens3 = f1_score(y_val, ens3_pred, average="macro")

print(f"Ensemble (BanglaBERT + mBERT) Macro F1:                {f1_ens2:.4f}")
print(f"Ensemble (BanglaBERT + mBERT + {best_embed_name}) Macro F1: {f1_ens3:.4f}")

candidates = {
    "BanglaBERT (fine-tuned)": f1_bangla,
    "mBERT (fine-tuned)": f1_mbert,
    "Ensemble (BanglaBERT+mBERT)": f1_ens2,
    f"Ensemble (BanglaBERT+mBERT+{best_embed_name})": f1_ens3,
}
BEST_STRATEGY = max(candidates, key=candidates.get)
print(f"\n>>> Best strategy on validation: {BEST_STRATEGY}  (Macro F1 = {candidates[BEST_STRATEGY]:.4f})")


## 8. Final prediction on the test set

In [ ]:
bangla_test_probs = predict_probs(bangla_trainer, bangla_ds_test)
mbert_test_probs = predict_probs(mbert_trainer, mbert_ds_test)
embed_test_probs = best_embed_clf.predict_proba(X_test_emb)

if BEST_STRATEGY == "BanglaBERT (fine-tuned)":
    final_test_probs = bangla_test_probs
elif BEST_STRATEGY == "mBERT (fine-tuned)":
    final_test_probs = mbert_test_probs
elif BEST_STRATEGY == "Ensemble (BanglaBERT+mBERT)":
    final_test_probs = weighted_soft_vote([bangla_test_probs, mbert_test_probs], [f1_bangla, f1_mbert])
else:
    final_test_probs = weighted_soft_vote(
        [bangla_test_probs, mbert_test_probs, embed_test_probs],
        [f1_bangla, f1_mbert, max(f1_lr, f1_svm, f1_lgbm)],
    )

final_test_pred = final_test_probs.argmax(axis=1)

submission = pd.DataFrame({"id": test_df["id"].values, "y_pred": final_test_pred})
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Saved /kaggle/working/submission.csv")
print(f"Final strategy used: {BEST_STRATEGY}")
submission.head(10)


## 9. Results summary

The cell below prints the final leaderboard of everything tried, and the predicted class
distribution on the test set (useful as a sanity check — it should roughly resemble the training
class distribution unless the test set was deliberately constructed otherwise).

In [ ]:
final_summary = pd.DataFrame(
    sorted(candidates.items(), key=lambda kv: -kv[1]) if 'candidates' in dir() else [],
    columns=["Strategy", "Val Macro F1"],
)
all_scores = {**results, **candidates}
final_summary = pd.DataFrame(sorted(all_scores.items(), key=lambda kv: -kv[1]), columns=["Model / Strategy", "Val Macro F1"])
final_summary["Val Macro F1"] = final_summary["Val Macro F1"].round(4)
print(final_summary.to_string(index=False))

print("\nPredicted test-set class distribution:")
print(pd.Series(final_test_pred).value_counts(normalize=True).sort_index().round(3))

print("\nTrue train-set class distribution (for comparison):")
print(train_df["label"].value_counts(normalize=True).sort_index().round(3))


## Notes on reproducibility & methodology (for submission writeup)

- **Data:** only the competition-provided train/test CSVs are used; no external data.
- **Preprocessing:** URL/HTML/mention stripping, emoji → text demojization, repeated
  character/punctuation normalization; Bangla Unicode and code-mixing preserved as-is.
- **Split:** single stratified 85/15 train/validation split, `random_state=42`, identical across
  every model for fair comparison.
- **Model 1 — embeddings + classical ML:** `paraphrase-multilingual-MiniLM-L12-v2` (frozen) →
  Logistic Regression / Linear SVM / LightGBM.
- **Model 2:** `csebuetnlp/banglabert`, fine-tuned end-to-end, 3-class head.
- **Model 3:** `bert-base-multilingual-cased`, fine-tuned end-to-end, 3-class head.
- **Shared transformer hyperparameters:** max_len=128, lr=2e-5, AdamW, batch size 16/32,
  up to 4 epochs with early stopping (patience 2) on validation Macro F1, weight decay 0.01,
  6% linear warmup, fp16 on GPU.
- **Ensembling:** soft-voting, per-model weights = that model's validation Macro F1.
- **Selection:** the strategy (single best model vs. 2-way vs. 3-way ensemble) with the highest
  validation Macro F1 is used to produce the final `submission.csv`.
- **Seed:** 42 everywhere (`random`, `numpy`, `torch`, HF `TrainingArguments(seed=...)`) for
  reproducibility; results may still vary slightly run-to-run due to GPU non-determinism in
  cuDNN kernels, which is expected and acceptable per the competition's reproducibility bar.
